# Praxisv03 Colab Runner

Use this notebook to run the next `praxisv03` Unraveled experiments on Colab GPU.

Recommended order:

1. Proper-budget `Mamba`
2. Graph `chunk128`
3. Graph `DE-weighted`


In [ ]:
REPO_SLUG = "garypagangit/praxis"
BRANCH = "main"
WORKSPACE_DIR = "/content/praxis-workspace"
DRIVE_ROOT = "/content/drive/MyDrive/praxis"

MAMBA_CONFIG = "configs/praxisv03-unraveled-colab-mamba-proper.json"
GRAPH_CHUNK128_CONFIG = "configs/praxisv03-unraveled-colab-graph-chunk128.json"
GRAPH_DE_WEIGHTED_CONFIG = "configs/praxisv03-unraveled-colab-graph-de-weighted.json"


## Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## Clone or Update The Repo

In [ ]:
import getpass
import os
import subprocess
import sys
from pathlib import Path

repo_url = f"https://github.com/{REPO_SLUG}.git"
token = getpass.getpass("GitHub token with read access to the private repo (leave blank only if the repo is public): ").strip()
if token:
    repo_url = f"https://{token}@github.com/{REPO_SLUG}.git"

workspace = Path(WORKSPACE_DIR)
if workspace.exists() and (workspace / ".git").exists():
    subprocess.run(["git", "remote", "set-url", "origin", repo_url], cwd=workspace, check=True)
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=workspace, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=workspace, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=workspace, check=True)
else:
    if workspace.exists():
        raise RuntimeError(f"{workspace} exists but is not a git repo. Remove it first.")
    subprocess.run(["git", "clone", "--branch", BRANCH, repo_url, str(workspace)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(workspace / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(workspace)], check=True)
os.chdir(workspace)
token = None
print("Workspace:", workspace)
subprocess.run([sys.executable, "-c", "import praxis; print(praxis.__file__)"] , check=True)


## Configure Persistent Runtime Paths

In [ ]:
import sys
from pathlib import Path

workspace = Path(WORKSPACE_DIR)
src_dir = workspace / "src"
if src_dir.exists() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

try:
    from praxis.colab_bootstrap import configure_persistent_runtime
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "Praxis is not installed in this Colab runtime yet. Run the 'Clone or Update The Repo' cell above first, make sure it finishes successfully, then rerun this cell."
    ) from exc

runtime_paths = configure_persistent_runtime(DRIVE_ROOT)
runtime_paths


## Verify GPU And Dataset

In [ ]:
import torch
from pathlib import Path

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

data_dir = Path("/content/drive/MyDrive/praxis/data/unraveled/network-flows")
print("Data dir exists:", data_dir.exists())
print("Sample contents:", sorted(p.name for p in list(data_dir.iterdir())[:5]))


## Run 1: Proper-Budget Mamba

In [ ]:
!python -m praxis.train --config {MAMBA_CONFIG}


## Run 2: Graph Chunk Fix (`flows_per_graph=128`)

In [ ]:
!python -m praxis.train --config {GRAPH_CHUNK128_CONFIG}


## Run 3: Graph DE-Weighted Loss

In [ ]:
!python -m praxis.train --config {GRAPH_DE_WEIGHTED_CONFIG}


## Quick Result Check

In [ ]:
from pathlib import Path

run_dirs = [
    Path("/content/drive/MyDrive/praxis/runs/praxisv03-unraveled-stage-balanced-colab-mamba-proper"),
    Path("/content/drive/MyDrive/praxis/runs/praxisv03-unraveled-stage-balanced-colab-graph-chunk128"),
    Path("/content/drive/MyDrive/praxis/runs/praxisv03-unraveled-stage-balanced-colab-graph-de-weighted"),
]

for run_dir in run_dirs:
    print("\n===", run_dir.name, "===")
    metrics_path = run_dir / "metrics-table.csv"
    if metrics_path.exists():
        print(metrics_path.read_text())
    else:
        print("metrics-table.csv not found yet")
